# VitaNexus-RX — Resume Full LightGBM Training
Run this notebook only after the repository changes are committed/pushed and the exported state plus immutable FAERS inputs are in Drive. LightGBM remains on its established CPU backend. This notebook never preprocesses FAERS and never enables fast mode.

## 1. Configuration — edit only this cell if your Drive root differs

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/VitaNexus-RX-ML'
REPOSITORY_URL = 'https://github.com/Sravanramaraju/VitaNexus-RX.git'
BRANCH = 'codex/faers-ml-clinical-integration'
REPOSITORY = '/content/VitaNexus-RX'
LOCAL_WORK = '/content/vitanexus-ml-work'
print({'driveRoot': DRIVE_ROOT, 'repository': REPOSITORY, 'branch': BRANCH, 'lightgbmBackend': 'CPU'})

## 2. Mount Google Drive — authorization appears here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Acquire the exact committed repository branch

In [ ]:
import pathlib, subprocess
repo = pathlib.Path(REPOSITORY)
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, REPOSITORY], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=repo, check=True)
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo, text=True).strip():
        raise RuntimeError('Existing Colab checkout is dirty; use a fresh runtime instead of discarding files.')
    subprocess.run(['git', 'checkout', BRANCH], cwd=repo, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=repo, check=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip())

## 4. Install reproducible Colab dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPOSITORY}/ml/requirements-colab.txt'], check=True)

## 5. Resolve and print Drive/local paths

In [ ]:
import sys
sys.path.insert(0, f'{REPOSITORY}/ml/colab')
from pathlib import Path
from colab_common import ColabPaths, configure
paths = ColabPaths(Path(DRIVE_ROOT), Path(REPOSITORY), Path(LOCAL_WORK))
configure(paths)

## 6. Verify Drive data and stage it to fast `/content` storage

In [ ]:
from colab_common import stage_dataset
stage_dataset(paths)

## 7. Import/verify old Windows state and run strict preflight

In [ ]:
from colab_common import lightgbm_preflight
preflight = lightgbm_preflight(paths, BRANCH)
assert 8 <= preflight['status']['bootstrap']['completed'] <= 20
assert preflight['status']['bootstrap']['total'] == 20

## 8. Resume LightGBM from interrupted replica 9

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'vitanexus_ml.cli', 'train-lightgbm'], cwd=REPOSITORY, env=os.environ.copy(), check=True)

## 9. Verify all 20 replicas, conformal evaluation and full promotion

In [ ]:
import json
from vitanexus_ml.cli import training_status
status = training_status()
assert status['status'] == 'COMPLETE', status
assert status['bootstrap']['completed'] == 20, status
manifest = json.loads((paths.drive_models / 'training_manifest.json').read_text())
assert manifest['fastMode'] is False and manifest['fullFinalData'] is True
for required in ('conformal_metrics.json', 'final_temporal_evaluation.json', 'lightgbm_metrics.json'):
    assert (paths.drive_reports / required).exists(), required
print('LIGHTGBM FULL TRAINING COMPLETE: 20/20 replicas, conformal and 2026 holdout reports verified')

## 10. Export a verified LightGBM-only inference bundle

In [ ]:
from vitanexus_ml.artifact_bundle import export_inference_bundle, verify_inference_bundle
output = paths.drive_exports / 'lightgbm_full_inference'
if (output / 'inference_bundle_manifest.json').exists():
    result = verify_inference_bundle(output, require_all=False)
else:
    result = export_inference_bundle(paths.drive_models, paths.drive_reports, output, component='lightgbm')
print(json.dumps(result, indent=2))

## 11. Final summary
Do not import the LightGBM-only bundle into the local application yet. Run the HGNN notebook next; it will create the combined final bundle required by local inference.